# 🛍 Customer Segmentation Analysis using RFM & K-Means Clustering
**Oasis Infobyte Internship — Data Analytics Track (Level 1 - Task 2)**  
**Author:** Oasis Intern  
**Repository:** `OIBSIP`

---

## 📌 1. Project Overview & Business Value
Customer Segmentation is the practice of dividing a company’s customer base into targeted cohorts sharing similar behavioral patterns. In this project, we implement the industry-standard **RFM (Recency, Frequency, Monetary)** framework coupled with unsupervised **K-Means Machine Learning Clustering** to discover behavioral personas and engineer targeted retention and monetization strategies.

### 📋 Feature Checklist:
- [x] Ingest e-commerce transactions, inspect schema, handle missing customer records
- [x] Calculate descriptive customer statistics (Average Order Value, Purchase Frequency, Customer Lifetime Value)
- [x] Engineer RFM features (Recency, Frequency, Monetary value)
- [x] Address feature skewness & apply `StandardScaler` normalization
- [x] Determine optimal cluster count $K$ using the **Elbow Method (Inertia)** and **Silhouette Score Analysis**
- [x] Train K-Means algorithm and assign granular behavioral segment personas
- [x] Visualize multidimensional clusters via 2D scatter plots and cohort bar charts
- [x] Formulate a targeted marketing action playbook per customer persona


In [ ]:
# Environment Setup & Dynamic Imports
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory for robust algorithm access
sys.path.insert(0, os.path.abspath('..'))

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
except Exception:
    from ml_core import StandardScaler, KMeans, silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

print("Dependencies successfully initialized.")


## 🔍 2. Data Ingestion & Data Hygiene
We load raw e-commerce transaction logs, drop records with missing Customer IDs, and parse timestamp attributes.


In [ ]:
df = pd.read_csv('data/ecommerce_transactions.csv')
print(f"Raw shape: {df.shape}")

# Drop rows missing customer identifier
df = df.dropna(subset=['Customer_ID']).copy()
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'])
print(f"Cleaned records: {len(df)} across {df['Customer_ID'].nunique()} unique customers.")
df.head()


## 📊 3. Descriptive Customer Statistics
Evaluating fundamental retail customer economics:
- **Average Order Value (AOV)**
- **Purchase Frequency**
- **Customer Lifetime Value (Historical Spend)**


In [ ]:
aov = df['Total_Spend'].mean()
cust_totals = df.groupby('Customer_ID')['Total_Spend'].sum()
cust_orders = df.groupby('Customer_ID')['Invoice_No'].nunique()

print(f"Average Order Value (AOV): ${aov:.2f}")
print(f"Average Purchase Frequency: {cust_orders.mean():.2f} orders/customer")
print(f"Average Customer Lifetime Value: ${cust_totals.mean():.2f}")
print(f"Maximum Customer Spend: ${cust_totals.max():.2f}")


## 📐 4. RFM Feature Engineering
We compute the 3 core dimensions of customer value:
1. **Recency ($R$)**: Number of days since the customer's last purchase relative to the snapshot reference date.
2. **Frequency ($F$)**: Total distinct invoice transactions generated by the customer.
3. **Monetary ($M$)**: Total gross dollar amount spent by the customer over the observation window.


In [ ]:
snapshot_date = df['Invoice_Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer_ID').agg({
    'Invoice_Date': lambda x: (snapshot_date - x.max()).days,
    'Invoice_No': 'nunique',
    'Total_Spend': 'sum'
}).reset_index()

rfm.rename(columns={
    'Invoice_Date': 'Recency',
    'Invoice_No': 'Frequency',
    'Total_Spend': 'Monetary'
}, inplace=True)

rfm.head()


## ⚖️ 5. Preprocessing & Feature Scaling
K-Means is a distance-based algorithm ($L_2$ Euclidean metric). To prevent monetary scale dominance and mitigate long-tailed skewness, we apply a log-transform followed by Z-score standardization (`StandardScaler`).


In [ ]:
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']].values)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print("Scaled feature shape:", rfm_scaled.shape)


## 🎯 6. Model Optimization: Elbow Method & Silhouette Evaluation
Evaluating $K \in [2, 8]$ to determine the optimal number of clusters where inertia diminishes and silhouette cohesion is maximized.


In [ ]:
inertias = []
sil_scores = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(list(k_range), inertias, marker='o', color='#2b5c8f', linewidth=2.5)
axes[0].set_title('Elbow Method for Optimal K (Inertia)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cluster Count (K)', fontsize=11)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=11)

axes[1].plot(list(k_range), sil_scores, marker='s', color='#27ae60', linewidth=2.5)
axes[1].set_title('Silhouette Score Evaluation', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster Count (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)

plt.tight_layout()
plt.show()


## 🚀 7. Final Clustering Model & Persona Assignment
Selecting $K = 4$ yields well-separated, interpretable commercial clusters.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().reset_index()
sorted_clusters = cluster_means.sort_values(by='Monetary', ascending=False)['Cluster'].tolist()

persona_map = {
    sorted_clusters[0]: 'Champions / High-Value VIPs',
    sorted_clusters[1]: 'Loyal Core Customers',
    sorted_clusters[2]: 'Recent / Potential Loyalists',
    sorted_clusters[3]: 'At-Risk / Hibernating'
}
rfm['Segment_Name'] = rfm['Cluster'].map(persona_map)

# Display profile summary table
profile_summary = rfm.groupby('Segment_Name').agg(
    Customer_Count=('Customer_ID', 'count'),
    Avg_Recency_Days=('Recency', 'mean'),
    Avg_Frequency_Orders=('Frequency', 'mean'),
    Avg_Monetary_Spend=('Monetary', 'mean'),
    Total_Segment_Revenue=('Monetary', 'sum')
).reset_index()

profile_summary


## 📊 8. Multidimensional Visualizations
Scatter projections illustrating customer separation across Recency, Frequency, and Monetary spend.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = {'Champions / High-Value VIPs': '#27ae60', 'Loyal Core Customers': '#2980b9', 
           'Recent / Potential Loyalists': '#f39c12', 'At-Risk / Hibernating': '#e74c3c'}

# Recency vs Monetary
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Segment_Name', palette=palette, alpha=0.8, s=60, ax=axes[0])
axes[0].set_title('Customer Clusters: Recency vs. Monetary Spend', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Recency (Days)', fontsize=11)
axes[0].set_ylabel('Total Spend ($)', fontsize=11)

# Frequency vs Monetary
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Segment_Name', palette=palette, alpha=0.8, s=60, ax=axes[1])
axes[1].set_title('Customer Clusters: Frequency vs. Monetary Spend', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency (Total Invoices)', fontsize=11)
axes[1].set_ylabel('Total Spend ($)', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Segment Distribution Bar Chart
plt.figure(figsize=(10, 5))
seg_counts = rfm['Segment_Name'].value_counts().reset_index()
seg_counts.columns = ['Segment_Name', 'Count']
sns.barplot(data=seg_counts, x='Segment_Name', y='Count', hue='Segment_Name', palette=palette, legend=False)
plt.title('Customer Count Across Behavioral Segments', fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Customer Segment', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.xticks(rotation=15)

for p in plt.gca().patches:
    plt.gca().annotate(f"{int(p.get_height())} ({p.get_height()/len(rfm)*100:.1f}%)", 
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


## 🎯 9. Segment-Specific Targeted Marketing Playbook
Based on the empirical customer clustering, the following specialized campaigns are deployed:

1. **🏆 Champions / High-Value VIPs**
   - **Characteristics**: Low recency, highest order frequency, highest aggregate spend.
   - **Playbook**: White-glove concierge customer service, exclusive early-access previews to product launches, high-tier loyalty perks. Avoid discount fatigue.
   
2. **💎 Loyal Core Customers**
   - **Characteristics**: Consistent transaction frequency, moderate-to-high monetary spend, moderate recency.
   - **Playbook**: Category cross-selling, anniversary milestone bonuses, invitation to referral rewards programs.

3. **🌱 Recent / Potential Loyalists**
   - **Characteristics**: Very low recency (recently active), moderate frequency, initial spend.
   - **Playbook**: Automated post-purchase nurture sequences, onboarding product education, time-limited second-purchase incentives.

4. **⚠️ At-Risk / Hibernating**
   - **Characteristics**: High recency (long inactivity), low historical order frequency.
   - **Playbook**: Automated 'We Miss You' win-back campaigns, feedback surveys offering 15% comeback coupons, and re-engagement push notifications.
